# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

## Rules

This notebook is checked automatically by **ThetaGrader (TG)**.

- Keep every `### Task N` heading intact -- TG uses it as a marker to locate tasks. Do not remove, rename, renumber, or duplicate these headings.
- Write your solution in the cell(s) reserved for it, directly below the corresponding task statement. Do not move a solution to a different place in the notebook or merge several tasks' solutions together.
- If a task asks for a written conclusion, report, or answer to a theoretical question, put it in a markdown cell directly under that task, clearly separated from code.
- Before submitting your final solution, make sure the `Status` field in the Bot-check cell below is set to `final`.

## AI usage disclosure

If you used AI assistants (e.g. ChatGPT, Claude, GitHub Copilot) while working on this homework, briefly describe here which tool(s) you used, for which parts of the assignment, and how.

Я использовал DeepSeek, чтобы писать бойлерплейт-код, такой как сохранение и загрузка чекпойнта и загрузка модели fasttext, а также искал справочную информацию по функциям pytorch и fasttext.

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
import transformers
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
    set_seed,
)
import datasets
from datasets import Dataset
from tqdm.auto import tqdm
import os

### Data Preparation

In [ ]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 70.8MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/7.83M [00:00<?, ?B/s]

test.jsonl: reconstructing file:   0%|          |  0.00B / 76.0MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [ ]:
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
CHECKPOINT_DIR = "modernbert_checkpoints"

In [ ]:
model_name = "answerdotai/ModernBERT-large"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name, dtype=torch.float32)

Loading weights: 100%|██████████| 172/172 [00:00<00:00, 16336.14it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [ ]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [ ]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[50281, 2347, 310, 253, 1495, 273, 247, 14168, 5974, 32, 16574, 368, 6266, 634, 1211, 8450, 32, 5028 ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=16, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=2
)

In [ ]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

### Task 1 (1 point):

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)

In [ ]:
from tqdm.auto import tqdm

model.to(DEVICE)
model.eval()

with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
  references, predictions = [], []
  for batch in tqdm(val_loader, desc='Validation'):
    labels = batch['labels']
    inputs = {key: value.to(DEVICE) for key, value in batch.items() if key != "labels"}
    predicted = model(**inputs).logits.argmax(dim=-1).cpu()
    predictions += predicted
    references += labels

predictions = torch.tensor(predictions, dtype=float)
references = torch.tensor(references, dtype=float)
accuracy = (predictions == references).sum() / len(predictions)

Validation:   0%|          | 0/2527 [00:00<?, ?it/s]

In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

### Task 2 (4 points):
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [ ]:
train_set = qqp_preprocessed["train"]
val_set = qqp_preprocessed["validation"]
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=128, shuffle=True, collate_fn=transformers.default_data_collator, num_workers=2
)
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=128, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=2
)

In [ ]:
EPOCHS = 3
LEARNING_RATE = 8e-5
SEED = 100

set_seed(SEED)

In [ ]:
model.to(DEVICE)

optimizer = torch.optim.AdamW(
  model.parameters(),
  lr=LEARNING_RATE,
  weight_decay=0.01
)

if DEVICE.type == "cuda":
  torch.cuda.synchronize()

In [ ]:
def evaluate(model, val_loader):
  model.eval()

  with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
    references, predictions = [], []
    for batch in tqdm(val_loader, desc='Validation'):
      labels = batch['labels']
      inputs = {key: value.to(DEVICE) for key, value in batch.items() if key != "labels"}
      predicted = model(**inputs).logits.argmax(dim=-1).cpu()
      predictions += predicted
      references += labels

  predictions = torch.tensor(predictions, dtype=float)
  references = torch.tensor(references, dtype=float)
  accuracy = (predictions == references).sum() / len(predictions)

  return accuracy

In [ ]:
scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

In [ ]:
%%time
for epoch in range(EPOCHS):
  model.train()
  total_loss, batches = 0.0, 0
  for batch in tqdm(train_loader, desc=f'Epoch {epoch + 1}'):
    inputs = {key: value.to(DEVICE) for key, value in batch.items()}
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
      loss = model(**inputs).loss
    if not torch.isfinite(loss):
      print(loss)
      break
    scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()
    total_loss += loss.item()
    batches += 1

  print({"epoch": epoch + 1, "train_loss": total_loss / batches, "val accuracy": evaluate(model, val_loader)})

  checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
  }
  # «Последний» чекпойнт — перезаписывается каждый раз
  torch.save(checkpoint, os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt'))

  if DEVICE.type == "cuda":
    torch.cuda.synchronize()

Validation: 100%|██████████| 316/316 [00:37<00:00,  8.33it/s]


{'epoch': 1, 'train_loss': 0.26339722084581874, 'val accuracy': tensor(0.8980)}


Validation: 100%|██████████| 316/316 [00:34<00:00,  9.06it/s]


{'epoch': 2, 'train_loss': 0.19189528080251828, 'val accuracy': tensor(0.9070)}


Validation: 100%|██████████| 316/316 [00:37<00:00,  8.38it/s]


{'epoch': 3, 'train_loss': 0.15268035447261008, 'val accuracy': tensor(0.9113)}
CPU times: user 48min 27s, sys: 3min 22s, total: 51min 49s
Wall time: 51min 45s


### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

### Task 3 (1 point):
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [ ]:
resume_path = os.path.join(CHECKPOINT_DIR, 'last_checkpoint.pt')

if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])

In [ ]:
def find_duplicate_questions(model, tokenizer, dataset, question):
  def preprocess(examples):
    result = tokenizer(
        examples["text1"],
        [question] * len(examples["text1"]),
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )
    return result

  preprocessed_dataset = dataset.map(preprocess, batched=True)
  dataloader = torch.utils.data.DataLoader(
    preprocessed_dataset, batch_size=64, shuffle=False, collate_fn=transformers.default_data_collator, num_workers=2
  )

  model.to(DEVICE)
  model.eval()
  with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16):
    all_probas = []
    for batch in tqdm(dataloader, desc='Prediction'):
      inputs = {key: value.to(DEVICE) for key, value in batch.items()}
      logits = model(**inputs).logits.float()
      probas = F.softmax(logits, dim=-1).cpu()[:, 1]
      all_probas.append(probas.cpu())

  all_probas = torch.cat(all_probas)
  candidate_probas, candidate_indices = torch.topk(all_probas, k=5)
  results = []
  for idx, p in zip(candidate_indices, candidate_probas):
    results.append({
        "idx": idx.item(),
        "question": preprocessed_dataset[idx.item()]["text1"],
        "score": float(p)
    })

  return results

In [ ]:
demo_set = qqp["train"]
df = demo_set.to_pandas()
df = df.drop_duplicates(subset=["text1"], keep="first")
df = df.drop_duplicates(subset=["text2"], keep="first")
print(df.shape)
demo_set = Dataset.from_pandas(df)
if "__index_level_0__" in demo_set.column_names:
    demo_set = demo_set.remove_columns("__index_level_0__")

(232422, 5)


In [ ]:
%%time
pd.DataFrame(find_duplicate_questions(model, tokenizer, demo_set, "Why did England lose Hundred Years' War?"))

Prediction: 100%|██████████| 3632/3632 [03:15<00:00, 18.53it/s]


CPU times: user 3min 33s, sys: 1.97 s, total: 3min 35s
Wall time: 3min 21s


,idx,question,score
0,124219,Why Central Govt banned old 500 and 1000 Rs no...,0.719384
1,148544,What do I do when Quora tells me to improve a ...,0.533641
2,22836,Why are some questions on Quora flagged as nee...,0.433981
3,203515,How do I get someone to stop harassing me on q...,0.432543
4,39234,Why has the Modi Government banned the 500 and...,0.339828


In [ ]:
pd.DataFrame(find_duplicate_questions(model, tokenizer, demo_set, "What is the formula for benzene?"))

Prediction: 100%|██████████| 4162/4162 [03:35<00:00, 19.31it/s]


,idx,question,score
0,8890,Why is Quora telling me my questions need impr...,0.619241
1,17147,Why did Indian government scrap Rs 1000 and 50...,0.544801
2,47304,Why does 500 and 1000 Rs notes banned by GOI a...,0.542863
3,159583,"Why PM Narendra Modi scrapped Rs 500, Rs 1000 ...",0.430147
4,89784,Why did Modi scrap Rs 500 & Rs 1000 notes? And...,0.379378


In [ ]:
pd.DataFrame(find_duplicate_questions(model, tokenizer, demo_set, "Who is stronger - Superman or Batman?"))

Prediction: 100%|██████████| 4162/4162 [03:27<00:00, 20.07it/s]


,idx,question,score
0,72345,Who is more powerful? Batman or Superman?,0.883034
1,8890,Why is Quora telling me my questions need impr...,0.578448
2,17147,Why did Indian government scrap Rs 1000 and 50...,0.458106
3,159583,"Why PM Narendra Modi scrapped Rs 500, Rs 1000 ...",0.409221
4,98198,Why do some completely straight forward questi...,0.362969


In [ ]:
pd.DataFrame(find_duplicate_questions(model, tokenizer, demo_set, "Who wrote War and Peace?"))

Prediction: 100%|██████████| 4162/4162 [03:27<00:00, 20.06it/s]


,idx,question,score
0,47304,Why does 500 and 1000 Rs notes banned by GOI a...,0.905239
1,89784,Why did Modi scrap Rs 500 & Rs 1000 notes? And...,0.882225
2,3179,How banning 500 and 1000 rupee can affect blac...,0.827271
3,141331,How banning 500 and 1000 rs note will help fig...,0.821331
4,17147,Why did Indian government scrap Rs 1000 and 50...,0.808219


In [ ]:
pd.DataFrame(find_duplicate_questions(model, tokenizer, demo_set, "Are energy drinks harmful?"))

Prediction: 100%|██████████| 4162/4162 [03:27<00:00, 20.06it/s]


,idx,question,score
0,8890,Why is Quora telling me my questions need impr...,0.792223
1,11061,Why are all my questions on Quora marked needi...,0.776369
2,47304,Why does 500 and 1000 Rs notes banned by GOI a...,0.759939
3,25662,Why does Quora say your question needs improve...,0.748398
4,27057,Why do all of my questions on Quora get marked...,0.746139


Если в датасете есть похожий вопрос, то модель выводит его (см. пример 3). Если же нет, то находятся в первую очередь вопросы с такими же ключевыми словами, многие из которых связаны с функциональностью платформы.

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

### Task 4 (Bonus, 0.5 point):
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

### Task 5 (Bonus, 1.5 points):
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

In [3]:
word2vec = api.load("fasttext-wiki-news-subwords-300")

[==================================================] 100.0% 958.5/958.4MB downloaded


In [15]:
def get_sentence_vector(text, word2vec, dim=300):
    words = simple_preprocess(text, min_len=1)
    vecs = [word2vec[w] for w in words if w in word2vec]
    if not vecs:
        return np.zeros(dim, dtype=np.float32)
    v = np.mean(vecs, axis=0).astype(np.float32)
    v_norm = np.linalg.norm(v) + 1e-9
    return v / v_norm

In [17]:
# идея: получаем эмбеддинги всего датасета и считаем косинусное расстояние их всех от эмбеддинга входного вопроса
# затем берём топ-5 по кос. расстоянию

def find_duplicate_questions_fasttext(model, dataset, question):
  embeddings = np.array([get_sentence_vector(t["text1"], model) for t in dataset], dtype=np.float32)
  question_embedding = get_sentence_vector(question, model)
  question_norm = np.linalg.norm(question_embedding) + 1e-9
  question_embedding /= question_norm

  similarities = embeddings @ question_embedding

  candidates = np.argpartition(-similarities, 5)[:5]
  candidates = candidates[np.argsort(-similarities[candidates])]
  results = []
  for idx in candidates:
    idx = int(idx)
    results.append({
        "idx": idx,
        "question": dataset[idx]["text1"],
        "score": similarities[idx]
    })

  return results

In [11]:
demo_set = qqp["train"]
df = demo_set.to_pandas()
df = df.drop_duplicates(subset=["text1"], keep="first")
df = df.drop_duplicates(subset=["text2"], keep="first")
print(df.shape)
demo_set = Dataset.from_pandas(df)
if "__index_level_0__" in demo_set.column_names:
    demo_set = demo_set.remove_columns("__index_level_0__")

(232422, 5)


In [18]:
%%time
pd.DataFrame(find_duplicate_questions_fasttext(word2vec, demo_set, "Why did England lose Hundred Years' War?"))

CPU times: user 43.5 s, sys: 160 ms, total: 43.7 s
Wall time: 44 s


,idx,question,score
0,77169,Why did the Soviet Union lose so many people d...,0.929030
1,123368,Why did the United States lose the Vietnam War?,0.922670
2,170226,Why did the Russians lose the Winter War?,0.922286
3,153551,Why did India lose the 1962 war against China?,0.920742
4,203675,Why did Poland receive so much historic German...,0.916354


In [20]:
pd.DataFrame(find_duplicate_questions_fasttext(word2vec, demo_set, "What is the formula for benzene?"))

,idx,question,score
0,96297,What is the formula for tetraphosphorus decasu...,0.981757
1,113625,What is the formula for acceleration?,0.978001
2,65399,What is the formula for hydrogen phosphate?,0.974753
3,222783,What is the formula for velocity?,0.974473
4,141095,What is the chemical formula for fermentation?,0.973486


In [21]:
pd.DataFrame(find_duplicate_questions_fasttext(word2vec, demo_set, "Who is stronger - Superman or Batman?"))

,idx,question,score
0,223808,Who is stronger: Superman or Supergirl?,0.983581
1,67267,Who is more powerful? Batman or Superman?,0.979037
2,127490,"Who is stronger, Goku or Superman?",0.968232
3,173477,Superheroes: Who is better: Batman or Iron Man?,0.961454
4,25035,"Who is smarter, Batman or Lex Luthor?",0.937992


In [22]:
pd.DataFrame(find_duplicate_questions_fasttext(word2vec, demo_set, "Who wrote War and Peace?"))

,idx,question,score
0,73803,Who said everything is fair in love and war?,0.893329
1,31421,Who won World War 2?,0.880418
2,164031,Who will win the war between India and france?,0.877970
3,661,Who joined the Korean war?,0.876518
4,197230,What did US soldiers who fought in the Vietnam...,0.873424


In [23]:
pd.DataFrame(find_duplicate_questions_fasttext(word2vec, demo_set, "Are energy drinks harmful?"))

,idx,question,score
0,133877,Are water retention pills dangerous?,0.874342
1,222524,What are some good uncarbonated drinks?,0.873893
2,29045,Are dark matter and dark energy: energy?,0.861635
3,66921,Are tortillas good for health? Are they fatten...,0.851837
4,135985,What are words for people who are greedy for s...,0.850496


Для решения этой задачи я использовал предобученные эмбеддинги FastText fasttext-wiki-news-subwords-300. Они имеют размерность 300 и обучены на данных Википедии и новостных сайтов. Их большое преимущество -- учёт не только существующих слов языка, но и буквенных N-грамм. Это позволяет корректно обрабатывать out-of-vocabulary-слова, что является преимуществом по сравнению с word2vec и GloVe. FastText работает на уровне слов, поэтому для получения эмбеддинга предложения мы усредняем эмбеддинги входящих в него слов (тех, которые есть в словаре модели).

Основной алгоритм такой:
1. Получить эмбеддинги всех вопросов из датасета (из поля text1) и отнормировать их.
2. Получить эмбеддинг входного вопроса и отнормировать его.
3. Посчитать косинусную близость каждой пары $(q, question) \;\;∀q ∈ embeddings$, где $question$ -- это входной запрос. Т.к. эмбеддинги всех вопросов уже нормированы, то можно заменить косинусную близость на скалярное произведение, что позволяет ускорить выполнение функции.
4. Найти топ-5 примеров с наибольшей косинусной близостью к question.


По сравнению с подходом на основе ModernBERT из задачи 3 решение на основе FastText выдаёт более семантически близкие запросы (например, на вопрос "Почему Англия проиграла в Столетней войне?" выдаются вопросы наподобие "Почему США проиграла Вьетнамскую войну?"), но не являющиеся дубликатами. В то же время ModernBERT, зафайнтюненный для поиска дубликатов, учитывает скорее синтаксическую близость вопросов (например, использование одинаковых вопросительных слов). В результате он выдаёт 1-2 очень похожих вопроса (те самые дубликаты), а остальная часть выдачи далека от содержания входного вопроса. Это объясняется тем, что  классификатор обучен искать именно дубликаты и присваивать им метку классификации 0/1, а не находить вопросы, похожие по семантике. В то же время косинусная близость позволяет оценить семантическое сходство и является непрерывной метрикой. Таким образом, решение на основе fasttext не является полноценным решением для поиска дубликатов, но лучше подходит для реального применения для выдачи рекомендаций на форумах и Q&A-платформах. Решение на основе fasttext также работает быстрее: на обработку одного входного вопроса уходит в среднем 44 с, а у ModernBERT -- 3.5 мин.